# ViFinQA — LLM chọn phép tính + dòng (Qwen3-8B)

Chạy trên Colab T4. Đọc `batch_*.jsonl`, ghi `decisions.jsonl`.

**Ràng buộc:** model phải < 14B tham số. Qwen3-8B = 8.2B ✓.
KHÔNG đổi sang Qwen3-14B (~14.7B) — vi phạm thể lệ.

**Bất biến (N7):** model chỉ trả về `operation` + `chosen` (chỉ số vào danh sách ứng viên) + `top_k`. Không bao giờ trả nhãn hay giá trị ô.

In [ ]:
# vllm==0.6.3 predates Qwen3 support entirely (no Qwen3ForCausalLM
# architecture registered) -- LLM(model="Qwen/Qwen3-8B") fails to load.
# Bump to a vLLM release new enough to have Qwen3 support.
#
# PINNED EXACTLY, not a floor: an open ">=0.8.5" resolves to whatever vLLM
# is newest the day this runs, and one such run pulled in cuda-python 13.3.1 /
# cuda-toolkit 13.0.3.0 -- newer than Colab's preinstalled RAPIDS stack
# (cuml/cudf/rmm/pylibraft all pin cuda-python<13.0, cuda-toolkit==12.*) and
# apparently newer than the T4 driver supports, so the vLLM engine subprocess
# crashed on startup with an unhelpful "Engine core initialization failed.
# Failed core proc(s): {}" and no further detail -- the real cause was in
# THIS cell's own pip conflict warnings, several cells before the crash.
#
# 0.8.5 itself later turned out uninstallable on this Colab image: it
# requires Python <3.13, and Colab was running 3.13
# (.../python3.13/dist-packages/... in the crash traceback). pip's own error
# listed every version it considered; 0.8.4 through 0.10.1.1 all declare the
# same <3.13 requirement, so 0.10.2 is the nearest version above 0.8.5 that
# installs at all here -- not verified yet to avoid the CUDA-13 problem
# above, since that requires a GPU to actually observe.
#
# CHECK BEFORE TRUSTING THIS CELL, every time it runs: scroll pip's output
# for a "cuda-python"/"cuda-toolkit" version conflict line (the ERROR block
# further down, after "dependency resolver does not currently take into
# account..."). If cuda-python/cuda-toolkit show a 13.x line again, stop --
# do not run the LLM(...) cell -- and report back rather than guessing at
# another pin.
#
# A broken run also leaves the wrong CUDA toolkit installed in this VM;
# `pip install` alone will not undo that. Runtime > Disconnect and delete
# runtime for a clean VM before re-running this cell.
!pip -q install "vllm==0.10.2" "huggingface_hub>=0.24"

In [ ]:
from google.colab import files
import pathlib

pathlib.Path("batches").mkdir(exist_ok=True)
print("Chọn toàn bộ batch_*.jsonl đã sinh bằng `submission row-batches`:")
uploaded = files.upload()
for name in uploaded:
    pathlib.Path("batches", name).write_bytes(uploaded[name])
print("đã nhận:", sorted(p.name for p in pathlib.Path("batches").glob("*.jsonl")))

In [ ]:
from vllm import LLM, SamplingParams

# Design doc (2026-08-22 Sec 5) specifies Qwen3-8B q4 (quantized), not
# full fp16. fp16 Qwen3-8B is ~16.4GB of weights alone -- already over a
# T4's 15GB VRAM budget before any KV cache, so dtype="half" at
# gpu_memory_utilization=0.90 OOMs before the first token.
# Loading a pre-quantized AWQ checkpoint (~5-6GB int4 weights) leaves
# headroom for the KV cache within budget. If this exact repo is
# unavailable, substitute any Qwen3-8B AWQ/GPTQ mirror and keep
# quantization="awq" (or "gptq") to match.
MODEL = "Qwen/Qwen3-8B-AWQ"  # 8.2B params < 14B. KHONG doi sang 14B.
llm = LLM(
    model=MODEL,
    quantization="awq",
    dtype="float16",
    gpu_memory_utilization=0.90,
    max_model_len=8192,
)
# max_tokens bumped from 16 -> 64 (see cell 4): Qwen3's chat template
# defaults to <think>...</think> reasoning content, which would consume
# the whole budget before any JSON is emitted.
# 64 was sized for a bare {"chosen_index": N} reply. A v2 decision is a
# whole JSON object, and llm.chat() below applies Qwen3's chat template,
# so give the reply room rather than truncating it into unparseable text.
sampling = SamplingParams(temperature=0.0, max_tokens=256)

In [ ]:
# llm.chat() (not llm.generate()) so vLLM applies Qwen3's chat template and the
# model is actually placed in the assistant role. Measured why: with raw
# completion, 1009/1012 decisions came back as parse()'s exact default triple
# (lookup, [0], None) -- an un-templated instruct model echoes the instructions,
# including the literal JSON skeleton, which is not valid JSON, so every reply
# fell through to the fallback and the run silently produced the rank-1 baseline.
#
# This cell only DEFINES things. The probe cell runs next, then the batch loop.
import json, pathlib, re

_OPERATIONS = {
    "lookup", "compare", "compare_companies", "difference",
    "growth_rate", "ratio", "average", "sum", "rank",
}

# The operation table below mirrors `plan_validator.py` arity exactly, because a
# decision the validator rejects does not fail loudly -- `assemble_plan`
# degrades it to `lookup` on the first chosen row and silently drops the rest.
# Measured on the first probe: all 3 questions came back "sum" when all 3 were
# plain lookups, because the old prompt never said lookup was the default nor
# that sum/average need a varying company or period. One of them asked to sum
# rows [10, 11]; the degrade path kept row 10 and threw row 11 away.
PROMPT = """Ban la tro ly phan tich bao cao tai chinh.
Khong giai thich, khong suy luan, khong dung <think>.

Cau hoi: {question}
Cong ty trong cau: {companies}
Ky trong cau: {periods}

Cac dong ung vien:
{candidates}

Chon operation theo bang duoi. MAC DINH LA "lookup" -- chi chon phep khac khi
cau hoi that su yeu cau tinh toan.

1 cong ty + 1 ky:
  lookup   -> hoi MOT so co san trong bang (dung cho HAU HET cau hoi)
  ratio    -> hoi TY LE / PHAN TRAM giua hai chi tieu -> chosen = [tu, mau]
  compare  -> hoi SO SANH hai chi tieu khac nhau      -> chosen = [a, b]

1 cong ty + 2 ky:
  difference  -> hoi CHENH LECH giua hai ky
  growth_rate -> hoi TANG TRUONG / phan tram thay doi
  -> chosen = [mot chi so dong chi tieu]

2 cong ty tro len:
  compare_companies -> so sanh dung 2 cong ty
  rank              -> hoi CAO NHAT / THAP NHAT / xep thu hang (can top_k)
  sum / average     -> hoi TONG / TRUNG BINH qua CAC CONG TY
  -> chosen = moi cong ty mot chi so, dung thu tu cong ty liet ke o tren

1 cong ty + nhieu ky:
  sum / average -> hoi TONG / TRUNG BINH qua CAC KY

QUY TAC BAT BUOC:
- Chi 1 cong ty va 1 ky -> KHONG duoc dung sum/average/rank/compare_companies.
  Dung lookup (hoac ratio/compare neu cau hoi hoi ty le / so sanh hai chi tieu).
- sum/average chi dung khi co NHIEU cong ty HOAC nhieu ky, khong phai de cong
  nhieu dong cua cung mot cong ty trong cung mot ky.
- top_k chi dien cho rank (1 = cao nhat, 2 = cao thu nhi). Con lai de null.

Chi tra ve JSON, khong kem gi khac:
{{"operation": "...", "chosen": [...], "top_k": null}}"""


def render(payload):
    lines = []
    for c in payload["candidates"]:
        parts = [f'[{c["index"]}] {c["row_label"]}']
        if c.get("company_code"):
            parts.append(f'(cong ty: {c["company_code"]})')
        if c.get("row_group_context"):
            parts.append(f'(muc: {c["row_group_context"]})')
        if c.get("table_title"):
            parts.append(f'(bang: {c["table_title"]})')
        if c.get("periods"):
            parts.append(f'(ky: {", ".join(c["periods"])})')
        lines.append(" ".join(parts))
    return PROMPT.format(
        question=payload["question"],
        companies=", ".join(payload.get("companies") or []) or "(khong ro)",
        periods=", ".join(payload.get("periods") or []) or "(khong ro)",
        candidates="\n".join(lines),
    )


def parse(text, limit, n_companies, n_periods):
    # Neu Qwen3 van ro ri <think>...</think>, bo han doan do truoc khi tim JSON --
    # neu khong, mot con so bat ky trong phan suy luan co the bi nhat nham.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    operation, chosen, top_k = "lookup", [], None
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            payload = json.loads(match.group(0))
            if payload.get("operation") in _OPERATIONS:
                operation = payload["operation"]
            raw = payload.get("chosen")
            if isinstance(raw, list):
                chosen = [int(v) for v in raw if isinstance(v, (int, float))]
            if isinstance(payload.get("top_k"), (int, float)):
                top_k = int(payload["top_k"])
        except (ValueError, TypeError):
            pass
    chosen = [v for v in chosen if 0 <= v < limit] or [0]

    # Enforce the same arity `plan_validator` will enforce. Doing it here keeps
    # the failure visible in this notebook's own counters; leaving it to
    # `assemble_plan` means a bad operation is silently downgraded to lookup on
    # `chosen[0]` with every other chosen row discarded.
    multi = n_companies > 1 or n_periods > 1
    if operation in {"rank", "compare_companies"} and n_companies < 2:
        operation = "lookup"
    if operation in {"sum", "average"} and not multi:
        operation = "lookup"
    if operation in {"difference", "growth_rate"} and n_periods < 2:
        operation = "lookup"
    if operation != "rank":
        top_k = None
    return {"operation": operation, "chosen": chosen, "top_k": top_k}


print("da dinh nghia PROMPT, render(), parse(). Chay cell tiep theo de kiem tra.")


In [ ]:
# CHAY CELL NAY TRUOC KHI CHAY VONG LAP. No in text tho cua model cho 5 cau
# dau tien, canh ket qua parse(), va dem xem bao nhieu cau roi vao gia tri mac
# dinh. Muc dich: bat loi trong vai giay thay vi sau mot tieng chay 16 batch.
#
# Lan 1: 1009/1012 la fallback vi dung raw completion (khong co chat template).
# Lan 2: 3/3 tra ve "sum" cho cau lookup vi prompt khong noi lookup la mac dinh.
# Ca hai lan deu chi lo ra SAU khi da chay xong. Cell nay ton tai de khong lap lai.
import json, pathlib

_probe_batch = sorted(pathlib.Path("batches").glob("batch_*.jsonl"))[0]
_probe = [json.loads(l) for l in _probe_batch.read_text("utf-8").splitlines() if l.strip()][:5]

_outs = llm.chat(
    [[{"role": "user", "content": render(p)}] for p in _probe],
    sampling,
    chat_template_kwargs={"enable_thinking": False},
)

_fallback = 0
for _p, _o in zip(_probe, _outs):
    _text = _o.outputs[0].text
    _n_comp = len(_p.get("companies") or [])
    _n_per = len(_p.get("periods") or [])
    _d = parse(_text, len(_p["candidates"]), _n_comp, _n_per)
    _is_fb = _d == {"operation": "lookup", "chosen": [0], "top_k": None}
    _fallback += _is_fb
    print("=" * 72)
    print("Q:", _p["question"][:95])
    print("cong ty:", _p.get("companies"), "| ky:", _p.get("periods"))
    print("--- model tra ve (text tho) ---")
    print(repr(_text[:300]))
    print("--- parse() ra ---", _d, "<-- FALLBACK" if _is_fb else "")

print()
print(f"fallback: {_fallback}/{len(_probe)}")
if _fallback == len(_probe):
    print("!!! TAT CA deu fallback -- DUNG LAI, dung chay vong lap.")
    print("Xem text tho o tren de biet model dang tra ve gi.")
else:
    print("OK. Kiem tra operation o tren co hop ly voi cau hoi khong, roi chay vong lap.")


In [ ]:
out = pathlib.Path("decisions.jsonl")
done = set()
if out.exists():  # chay lai sau timeout chi ton phan con thieu
    done = {json.loads(l)["question_id"] for l in out.read_text("utf-8").splitlines() if l.strip()}
    print("da co san:", len(done))

with out.open("a", encoding="utf-8") as sink:
    for batch in sorted(pathlib.Path("batches").glob("batch_*.jsonl")):
        payloads = [json.loads(l) for l in batch.read_text("utf-8").splitlines() if l.strip()]
        payloads = [p for p in payloads if p["question_id"] not in done and p["candidates"]]
        if not payloads:
            continue
        outputs = llm.chat(
            [[{"role": "user", "content": render(p)}] for p in payloads],
            sampling,
            chat_template_kwargs={"enable_thinking": False},
        )
        for payload, output in zip(payloads, outputs):
            decision = parse(
                output.outputs[0].text,
                len(payload["candidates"]),
                len(payload.get("companies") or []),
                len(payload.get("periods") or []),
            )
            sink.write(json.dumps({"question_id": payload["question_id"], **decision}) + "\n")
        sink.flush()
        print(batch.name, "xong", len(payloads), "cau")

import collections
_all = [json.loads(l) for l in out.read_text("utf-8").splitlines() if l.strip()]
print()
print("tong quyet dinh:", len(_all))
print("phan bo operation:", collections.Counter(d["operation"] for d in _all).most_common())
_fb = sum(1 for d in _all if d["operation"] == "lookup" and d["chosen"] == [0] and d["top_k"] is None)
print(f"trung gia tri mac dinh cua parse(): {_fb}/{len(_all)}")
print("(neu con so nay gan bang tong, model khong thuc su tra loi -- xem lai truoc khi tai ve)")


In [ ]:
from google.colab import files
files.download("decisions.jsonl")